# Distributed Partition Function (DPF) — Example Walkthrough

This notebook demonstrates how to use the **Distributed Partition Function (DPF)** to process data in parallel across multiple nodes in a compute pool. DPF partitions your data and executes your Python function on each partition concurrently, handling distributed orchestration, errors, observability, and artifact persistence automatically.

We'll use a **supply chain allocation** scenario as the example: given factories with limited capacity and warehouses with specific demand, find the optimal shipping plan per region using `scipy.optimize.linprog`.

DPF supports two execution modes:

- **DataFrame mode** (`run()`): Partition a Snowpark DataFrame by column values and execute your function on each partition concurrently.
- **Stage mode** (`run_from_stage()`): Process files from a Snowflake stage where each file becomes a partition. Ideal for large-scale file processing with predictable memory usage.

**Environment:** This notebook is designed to run in a Snowflake Notebook on Container Runtime. If running locally, see the **ML Jobs deployment** section at the bottom.

**Prerequisites:**
- A compute pool with max nodes >= 3 (e.g., `CPU_X64_S`), or the system-provided `SYSTEM_COMPUTE_POOL_CPU`

---
## Setup

In [ ]:
from datetime import datetime
import json

import pandas as pd
import numpy as np

from snowflake.snowpark import Session


session = Session.builder.getOrCreate()

# Configuration
database = session.get_current_database() or "MY_DATABASE" # Change to your database
schema = session.get_current_schema() or "MY_SCHEMA" # Change to your schema

input_stage = "DPF_INPUT_STAGE"
dpf_stage = "DPF_RESULTS_STAGE"
input_table = "SUPPLY_CHAIN_DATA"
output_table = "OPTIMIZED_SHIPPING_MANIFEST"

# Create stages
session.use_schema(f"{database}.{schema}")
session.sql(f"CREATE STAGE IF NOT EXISTS {dpf_stage}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS {input_stage}").collect()

print(f"Session: {session}")

### Import DPF modules and Scale Compute Nodes
Snowflake Notebook on Container Runtime only — skip this cell if running locally.

In [ ]:
from snowflake.ml.modeling.distributors.distributed_partition_function.dpf import DPF
from snowflake.ml.modeling.distributors.distributed_partition_function.dpf_run import (
    DPFRun,
)
from snowflake.ml.modeling.distributors.distributed_partition_function.entities import (
    RunStatus,
    ExecutionOptions,
)
from snowflake.ml.runtime_cluster import scale_cluster

# Scale to 3 nodes for parallel processing
scale_cluster(expected_cluster_size=3)

### Create Synthetic Supply Chain Data

Generate a dataset with 5 regions, each containing 3 factories (supply) and 10 warehouses (demand).

In [ ]:
def create_supply_chain_data(session, table_name):
    """Generate synthetic supply chain data with factories and warehouses across regions."""
    regions = ["NA_WEST", "NA_EAST", "EMEA_CENTRAL", "APAC_SOUTH", "LATAM"]
    np.random.seed(42)
    data = []

    for reg in regions:
        # 3 Factories per region (supply)
        for i in range(3):
            data.append(
                {
                    "REGION": reg,
                    "LOCATION_ID": f"FACT_{reg}_{i}",
                    "TYPE": "FACTORY",
                    "LAT": np.random.uniform(25, 55),
                    "LON": np.random.uniform(-130, -60),
                    "CAPACITY": 1000,
                    "DEMAND": 0,
                }
            )
        # 10 Warehouses per region (demand)
        for j in range(10):
            data.append(
                {
                    "REGION": reg,
                    "LOCATION_ID": f"WH_{reg}_{j}",
                    "TYPE": "WAREHOUSE",
                    "LAT": np.random.uniform(25, 55),
                    "LON": np.random.uniform(-130, -60),
                    "CAPACITY": 0,
                    "DEMAND": 250,
                }
            )

    df = pd.DataFrame(data)
    sdf = session.create_dataframe(df)
    sdf.write.mode("overwrite").save_as_table(table_name)
    print(f"Created '{table_name}' with {len(df)} rows across {len(regions)} regions")
    return session.table(table_name)


supply_chain_sdf = create_supply_chain_data(session, input_table)
supply_chain_sdf.show()

---
## DataFrame Mode: Process Data by Column Partitions

Partition the `SUPPLY_CHAIN_DATA` table by `REGION` and solve each region's allocation in parallel.

1. **Define the processing function** — optimization logic that runs on each partition.
2. **Initialize and run DPF** — launch parallel execution across all partitions.
3. **Monitor progress** — track status and wait for completion.
4. **Retrieve results** — collect artifacts and output data from each partition.
5. **Restore a completed run** — access results from a previous run without re-executing.

### Step 1: Define the Processing Function

This function runs on each partition (region). It receives the partition's data via `data_connector` and
uses `scipy.optimize.linprog` to solve the transportation problem: minimize shipping cost while
satisfying warehouse demand without exceeding factory capacity.

In [ ]:
def solve_allocation(data_connector, context):
    """
    Solve the supply chain allocation problem for a single region.

    Uses linear programming to find the optimal shipment plan that minimizes
    total transportation cost (based on distance) subject to:
      - Factory capacity constraints (supply)
      - Warehouse demand constraints (demand)

    Args:
        data_connector: Provides access to the partition's data.
        context: PartitionContext with partition_id and artifact methods.
    """
    from scipy.optimize import linprog
    from scipy.spatial.distance import cdist
    import pandas as pd
    import numpy as np
    import json

    df = data_connector.to_pandas()
    region = context.partition_id
    print(f"[{region}] Processing {len(df)} locations")

    factories = df[df["TYPE"] == "FACTORY"].reset_index(drop=True)
    warehouses = df[df["TYPE"] == "WAREHOUSE"].reset_index(drop=True)
    n_fact = len(factories)
    n_wh = len(warehouses)

    # Build cost matrix (Euclidean distance as proxy for shipping cost)
    cost_matrix = cdist(
        factories[["LAT", "LON"]], warehouses[["LAT", "LON"]], metric="euclidean"
    )
    c = cost_matrix.flatten()

    # Inequality constraint: total outbound from Factory_i <= Capacity_i
    A_ub = np.zeros((n_fact, n_fact * n_wh))
    for i in range(n_fact):
        A_ub[i, i * n_wh : (i + 1) * n_wh] = 1
    b_ub = factories["CAPACITY"].values.astype(float)

    # Equality constraint: total inbound to Warehouse_j == Demand_j
    A_eq = np.zeros((n_wh, n_fact * n_wh))
    for j in range(n_wh):
        A_eq[j, j::n_wh] = 1
    b_eq = warehouses["DEMAND"].values.astype(float)

    # Solve
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, method="highs")

    if res.success:
        allocation = res.x.reshape((n_fact, n_wh))
        manifest = []
        for i in range(n_fact):
            for j in range(n_wh):
                qty = allocation[i, j]
                if qty > 0.1:
                    manifest.append(
                        {
                            "REGION": region,
                            "ORIGIN": factories.loc[i, "LOCATION_ID"],
                            "DESTINATION": warehouses.loc[j, "LOCATION_ID"],
                            "SHIPMENT_QTY": round(float(qty), 2),
                            "UNIT_DISTANCE": round(float(cost_matrix[i, j]), 4),
                        }
                    )

        manifest_df = pd.DataFrame(manifest)

        summary = {
            "region": region,
            "status": "OPTIMAL",
            "total_cost": round(float(res.fun), 2),
            "shipment_count": len(manifest),
            "total_units_shipped": round(sum(m["SHIPMENT_QTY"] for m in manifest), 2),
        }
        print(
            f"[{region}] Optimal cost: {summary['total_cost']}, shipments: {len(manifest)}"
        )

        # Upload summary as a stage artifact
        context.upload_to_stage(
            summary,
            "summary.json",
            write_function=lambda obj, path: json.dump(obj, open(path, "w")),
        )

        # Write results to a Snowflake table using the bounded session pool
        context.with_session(
            lambda session: session.create_dataframe(manifest_df)
            .write.mode("append")
            .save_as_table(output_table)
        )
    else:
        print(f"[{region}] Optimization failed: {res.message}")

### Step 2: Initialize and Run DPF

In [ ]:
dpf = DPF(func=solve_allocation, stage_name=dpf_stage)

session.sql(f"DROP TABLE IF EXISTS {output_table}").collect()

run = dpf.run(
    partition_by="REGION",
    snowpark_dataframe=session.table(input_table),
    run_id=f"supply_chain_{datetime.now():%Y%m%d_%H%M%S}",
    execution_options=ExecutionOptions(use_head_node=True, num_cpus_per_worker=1),
)
print(f"Launched: {run.run_id}")

### Step 3: Monitor Progress and Wait for Completion

In [ ]:
final_status = run.wait()  # Shows progress
print(f"Job completed with status: {final_status}")

In [ ]:
# Quick summary of all partition statuses
progress = run.get_progress()
for status, partitions in progress.items():
    print(f"{status}: {len(partitions)} partitions")

### Step 4: Retrieve Results from Each Partition

In [ ]:
def print_results(summaries):
    """Format and display the supply chain optimization results."""
    for s in summaries:
        print(f"  {s['region']}: cost={s['total_cost']}, shipments={s['shipment_count']}")

    total_cost = sum(s["total_cost"] for s in summaries)
    total_shipments = sum(s["shipment_count"] for s in summaries)
    print(f"\n  TOTAL: cost={total_cost:.2f}, shipments={total_shipments}")

In [ ]:
if final_status == RunStatus.SUCCESS:
    summaries = []
    for partition_id, details in run.partition_details.items():
        files = details.stage_artifacts_manager.list()
        print(f"Partition '{partition_id}' artifacts: {files}")

        summary = details.stage_artifacts_manager.get(
            "summary.json",
            read_function=lambda path: json.load(open(path, "r")),
        )
        summaries.append(summary)

    print_results(summaries)
else:
    print(f"Run did not succeed: {final_status}")

In [ ]:
# View the results written to the Snowflake table
session.table(output_table).show()

### Inspect Partition Logs

View stdout/stderr from individual partitions to verify processing or debug failures.

In [ ]:
# View logs from each partition
for partition_id, details in run.partition_details.items():
    print(f"--- {partition_id} ---")
    print(details.logs)

In [ ]:
# Debug failed partitions (if any)
# progress = run.get_progress()
# for partition in progress.get("FAILED", []):
#     print(f"--- Failed: {partition.partition_id} ---")
#     print(partition.logs)

### Step 5: Restore Results from a Completed Run

Access results from a previous run without re-executing. Useful after restarting a notebook session.

In [ ]:
restored_run = DPFRun.restore_from(
    run_id=run.run_id,
    stage_name=dpf_stage,
)

print(f"Restored run status: {restored_run.status}")
for partition_id, details in restored_run.partition_details.items():
    print(f"  {partition_id}: {details.status}")

# Note: Restored runs are read-only. You cannot call wait() or cancel() on them.

---
## Stage Mode: Process Files from a Stage

Process pre-staged parquet files where each file becomes a partition.
First, copy the data from the table to stage as parquet files, one per region.

In [ ]:
# Prepare parquet files on stage — one file per region
session.sql(f"REMOVE @{input_stage}/supply_chain/").collect()

session.sql(
    f"""
    COPY INTO @{input_stage}/supply_chain/
    FROM {input_table}
    PARTITION BY REGION
    FILE_FORMAT = (TYPE = PARQUET COMPRESSION = SNAPPY)
    MAX_FILE_SIZE = 15728640
    HEADER = TRUE
"""
).collect()

# Verify staged files
session.sql(f"LIST @{input_stage}/supply_chain/").show()

### Run DPF from Stage

The processing function signature is the same as DataFrame mode. The `data_connector` provides access
to each file's data, and `context.partition_id` is the relative file path.

In [ ]:
dpf_from_stage = DPF(func=solve_allocation, stage_name=dpf_stage)

session.sql(f"DROP TABLE IF EXISTS {output_table}").collect()

stage_run = dpf_from_stage.run_from_stage(
    stage_location=f"@{input_stage}/supply_chain/",
    run_id=f"supply_chain_stage_{datetime.now():%Y%m%d_%H%M%S}",
    file_pattern="*.parquet",
    execution_options=ExecutionOptions(
        use_head_node=True,
        num_cpus_per_worker=1,
    ),
)
print(f"Launched: {stage_run.run_id}")

In [ ]:
stage_status = stage_run.wait()
print(f"Stage mode completed with status: {stage_status}")

In [ ]:
# View the results written to the Snowflake table
session.table(output_table).show()

---
## Deploy with ML Jobs via `@remote`

Run DPF in an ML Job from any IDE. ML Jobs execute inside Snowpark Container Services
and can scale across multiple nodes. Logs are available in Snowsight under Monitoring > Services & Jobs.

In [ ]:
job_stage = "DPF_JOB_STAGE"
compute_pool = "SYSTEM_COMPUTE_POOL_CPU"  # Update with your compute pool name

session.sql(f"CREATE STAGE IF NOT EXISTS {job_stage}").collect()

In [ ]:
from snowflake.ml.jobs import remote

@remote(
    compute_pool=compute_pool,
    stage_name=job_stage,
    target_instances=3,
)
def launch_supply_chain_job():
    """
    Launch a DPF supply chain optimization run as an ML Job.
    """
    from datetime import datetime
    from snowflake.snowpark import Session
    from snowflake.ml.modeling.distributors.distributed_partition_function.dpf import (
        DPF,
    )
    from snowflake.ml.modeling.distributors.distributed_partition_function.entities import (
        ExecutionOptions,
    )

    session = Session.builder.getOrCreate()
    dpf_input = session.table(input_table)

    dpf = DPF(func=solve_allocation, stage_name=dpf_stage)
    run = dpf.run(
        partition_by="REGION",
        snowpark_dataframe=dpf_input,
        run_id=f"job_supply_chain_{datetime.now():%Y%m%d_%H%M%S}",
        execution_options=ExecutionOptions(use_head_node=False),
    )
    run.wait()

    print(f"DPF run complete: {run.run_id}")
    return run.run_id

session.sql(f"DROP TABLE IF EXISTS {output_table}").collect()

job = launch_supply_chain_job()
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")

In [ ]:
# Check the status and logs of the ML Job
print(f"Status: {job.status}")
print(job.get_logs())

In [ ]:
# View the results written to the Snowflake table
session.table(output_table).show()

---
## Cleanup

Scale the cluster back down to a single node when you're done.

In [ ]:
scale_cluster(expected_cluster_size=1)

# Uncomment to drop objects created by this notebook
# session.sql(f"DROP TABLE IF EXISTS {input_table}").collect()
# session.sql(f"DROP TABLE IF EXISTS {output_table}").collect()
# session.sql(f"DROP STAGE IF EXISTS {dpf_stage}").collect()
# session.sql(f"DROP STAGE IF EXISTS {input_stage}").collect()
# session.sql(f"DROP STAGE IF EXISTS {job_stage}").collect()